### Attribution
Sibling notebook to `nps.ipynb` in this folder. Both are adapted from:
https://github.com/miptgirl/miptgirl_medium/blob/main/dspy_example/nps_topic_modelling.ipynb

**This variant** replaces `dspy.MIPROv2` with `dspy.GEPA` for the first prompt-optimisation step. The `dspy.BootstrapFewShotWithRandomSearch` section later in the notebook is unchanged, so the two optimisers can be compared side-by-side on the same NPS topic-classification task.

In [1]:
import pandas as pd
import tqdm

### Try out DSPy on a simple example

In [4]:

SMALL_MODEL_CANDIDATES: list[str] = [
    "gemini/gemini-2.5-flash-lite",
    "gemini/gemini-2.5-flash",
    "gemini/gemini-2.0-flash",
]

REFLECTION_MODEL_CANDIDATES: list[str] = [
    "gemini/gemini-3.1-pro-preview",
    "gemini/gemini-2.5-pro",
    "gemini/gemini-1.5-pro",
]
import dspy
llm = dspy.LM(SMALL_MODEL_CANDIDATES[0])
dspy.configure(lm=llm)
dspy.configure_cache(enable_memory_cache=False, enable_disk_cache=False)

In [5]:
simple_model = dspy.Predict("question -> answer: int", cache = False)
simple_model(question="I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?")

Prediction(
    answer=5
)

In [6]:
dspy.inspect_history(n = 1)





[2026-05-11T22:54:47.738733]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be a single int value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?

Respond with the corresponding output fields, starting with the field `[[ ## answer ## ]]` (must be formatted as a valid Python int), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## answer ## ]]
5
[[ ## completed ## ]]







In [8]:
from utils import wrap_text

cot_model = dspy.ChainOfThought("question -> answer: int")
answer = cot_model(question="I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?")

print(f"Answer: {answer.answer}\n")
print("Reasoning:")
print(wrap_text(answer.reasoning, width=80))

dspy.inspect_history(n = 1)





Answer: 5

Reasoning:
This is a combination problem because the order in which the balls are selected
does not matter. We need to find the number of ways to choose 4 balls out of 5
distinct balls. The formula for combinations is given by C(n, k) = n! / (k! *
(n-k)!), where n is the total number of items and k is the number of items to
choose. In this case, n = 5 (total number of balls) and k = 4 (number of balls
to select).

C(5, 4) = 5! / (4! * (5-4)!)
C(5, 4) = 5! / (4! * 1!)
C(5, 4) = (5 * 4 * 3 * 2 * 1) / ((4 * 3 * 2 * 1) * 1)
C(5, 4) = 120 / (24 * 1)
C(5, 4) = 120 / 24
C(5, 4) = 5

Alternatively, choosing 4 balls out of 5 is the same as *not* choosing 1 ball
out of 5. So, C(5, 4) = C(5, 5-4) = C(5, 1).
C(5, 1) = 5! / (1! * (5-1)!)
C(5, 1) = 5! / (1! * 4!)
C(5, 1) = (5 * 4 * 3 * 2 * 1) / (1 * (4 * 3 * 2 * 1))
C(5, 1) = 5

Therefore, there are 5 possible combinations of the balls.




[2026-05-11T23:04:57.328667]

System message:

Your input fields are:
1. `question` (str):
Your out

In [9]:
dspy.inspect_history(n = 1)





[2026-05-11T23:04:57.328667]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be a single int value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## answer ## ]]` (must be formatted as a valid Python int), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
This is a combination problem because

In [10]:
dspy.configure(adapter=dspy.JSONAdapter())

In [11]:
response = cot_model(question="I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?")
print(f"Reasoning:\n {wrap_text(response['reasoning'])}")
print(f"Answer: {response['answer']}")

dspy.inspect_history(n = 1)





Reasoning:
 The problem asks for the number of combinations of selecting 4 balls from a set
of 5 distinct balls. This is a combination problem, as the order in which the
balls are selected does not matter. The formula for combinations is C(n, k) = n!
/ (k! * (n-k)!), where n is the total number of items to choose from, and k is
the number of items to choose. In this case, n = 5 and k = 4. So, C(5, 4) = 5! /
(4! * (5-4)!) = 5! / (4! * 1!) = (5 * 4 * 3 * 2 * 1) / ((4 * 3 * 2 * 1) * 1) = 5
/ 1 = 5.
Answer: 5




[2026-05-11T23:05:15.091167]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## question ## ]]
{question}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}        # note: the value you produce must 

### JSONAdapter

In [12]:
dspy.configure(adapter=dspy.JSONAdapter())
print(cot_model(question="I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?"))

Prediction(
    reasoning='This is a combination problem, as the order in which the balls are selected does not matter. We need to find the number of combinations of selecting 4 balls from a set of 5. The formula for combinations is C(n, k) = n! / (k! * (n-k)!), where n is the total number of items to choose from, and k is the number of items to choose. In this case, n = 5 and k = 4. So, C(5, 4) = 5! / (4! * (5-4)!) = 5! / (4! * 1!) = (5 * 4 * 3 * 2 * 1) / ((4 * 3 * 2 * 1) * 1) = 120 / (24 * 1) = 120 / 24 = 5. Therefore, there are 5 possible combinations.',
    answer=5
)


In [13]:
dspy.inspect_history(n = 1)





[2026-05-11T23:05:28.272405]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## question ## ]]
{question}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}        # note: the value you produce must be a single int value"
}
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?

Respond with a JSON object in the following order of fields: `reasoning`, then `answer` (must be formatted as a valid Python int).


Response:

{
  "reasoning": "This is a combination problem, as the order in which the balls 

In [15]:
response = cot_model(question="I have 25 different balls and I randomly select 9. How many possible combinations of the balls I can get?")
print(f"Reasoning:\n {wrap_text(response['reasoning'])}")
print(f"Answer: {response['answer']}")
dspy.inspect_history(n = 1)


Reasoning:
 This is a combination problem. We need to find the number of ways to choose 9
balls from a set of 25 distinct balls. The formula for combinations is C(n, k) =
n! / (k!(n-k)!), where n is the total number of items and k is the number of
items to choose. In this case, n = 25 and k = 9. So, C(25, 9) = 25! /
(9!(25-9)!) = 25! / (9!16!). Calculating this value: 25*24*23*22*21*20*19*18*17
/ (9*8*7*6*5*4*3*2*1) = 2,042,975.
Answer: 2042975




[2026-05-11T23:05:48.371465]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (int):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## question ## ]]
{question}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}        # note: the value you produce must be a single int value"
}
In adhering to this structure, your o

#### Cross-check

In [16]:
import math
n = 25
k = 9
round(math.factorial(n)/math.factorial(k)/math.factorial(n-k))
2042975

2042975

In [17]:
pip install deno

/Users/aurobindotripathy/prompt-opt-cookbook/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
from dspy import PythonInterpreter
def evaluate_math(expr: str) -> str:
    # Executes Python and returns the output as string
    with PythonInterpreter() as interp:
        return interp(expr)

react_model = dspy.ReAct(
    signature="question -> answer: int",  # expects an int answer
    tools=[evaluate_math]
)
response = react_model(question="I have 25 different balls and I randomly select 9. How many possible combinations of the balls I can get?")
print(response)
response.answer

2026/05/11 23:06:05 WARNING dspy.primitives.python_interpreter: Unable to find the Deno cache dir.


Prediction(
    trajectory={'thought_0': 'The user is asking for the number of combinations of selecting 9 balls out of 25. This is a combinatorics problem that can be solved using the combination formula: C(n, k) = n! / (k! * (n-k)!), where n is the total number of items, and k is the number of items to choose. In this case, n=25 and k=9. I should use the `evaluate_math` tool to calculate this.', 'tool_name_0': 'evaluate_math', 'tool_args_0': {'expr': '25! / (9! * (25-9)!)'}, 'observation_0': 'Execution error in evaluate_math: \nTraceback (most recent call last):\n  File "/Users/aurobindotripathy/prompt-opt-cookbook/.venv/lib/python3.11/site-packages/dspy/primitives/python_interpreter.py", line 322, in _ensure_deno_process\n    self.deno_process = subprocess.Popen(\n                        ^^^^^^^^^^^^^^^^^\n  File "/Users/aurobindotripathy/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/lib/python3.11/subprocess.py", line 1026, in __init__\n    self._execute_child(args, exe

0

In [19]:
response.trajectory

{'thought_0': 'The user is asking for the number of combinations of selecting 9 balls out of 25. This is a combinatorics problem that can be solved using the combination formula: C(n, k) = n! / (k! * (n-k)!), where n is the total number of items, and k is the number of items to choose. In this case, n=25 and k=9. I should use the `evaluate_math` tool to calculate this.',
 'tool_name_0': 'evaluate_math',
 'tool_args_0': {'expr': '25! / (9! * (25-9)!)'},
 'observation_0': 'Execution error in evaluate_math: \nTraceback (most recent call last):\n  File "/Users/aurobindotripathy/prompt-opt-cookbook/.venv/lib/python3.11/site-packages/dspy/primitives/python_interpreter.py", line 322, in _ensure_deno_process\n    self.deno_process = subprocess.Popen(\n                        ^^^^^^^^^^^^^^^^^\n  File "/Users/aurobindotripathy/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/lib/python3.11/subprocess.py", line 1026, in __init__\n    self._execute_child(args, executable, preexec_fn, clo

# NPS
**Definition**:Net Promoter Score (NPS) is a customer loyalty metric that measures the likelihood of customers recommending a company, ranging from -100 to 100. It is calculated by subtracting the percentage of detractors (0–6 score) from promoters (9–10 score). A score above 0 is good, 50+ is excellent, and 70+ is world-class


In [20]:
import json
with open('nps_comments.json', 'r') as f:
    nps_data = json.loads(f.read())

print(f"sample:\n {nps_data[0]}")

sample:
 {'topics': ['Limited Size or Shade Availability'], 'comment': "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?"}


#### Get all topics

In [21]:
topics = set()

for rec in nps_data: 
    for t in rec['topics']: 
        topics.add(t)

In [22]:
topic_list = list(topics)
print(wrap_text(str(topic_list)))
print(f"Topic count: {len(topic_list)}")

['Unresponsive or Generic Customer Support', 'Inaccurate Product Descriptions or
Photos', 'Complicated Returns or Exchanges', 'Difficult Product Discovery',
'Slow or Unreliable Shipping', 'Customs and Import Charges', 'Limited Size or
Shade Availability', 'Confusing Loyalty or Discount Systems', 'Website or App
Bugs', 'Damaged or Incorrect Items']
Topic count: 10


#### Prompt Optimization with GEPA

GEPA (Genetic-Pareto Evolving Prompt Agents) optimises a DSPy program by using a separate **reflection LM** to analyse failures, propose improved instructions, and evolve the prompt across a Pareto front of candidates. The reflection LM is typically the strongest model available since it must reason about *why* outputs fail; here we point it at the top entry in `REFLECTION_MODEL_CANDIDATES`.

In [23]:
from typing import Literal, List

class NPSTopic(dspy.Signature):
    """Classify NPS topics"""

    comment: str = dspy.InputField()
    answer: List[Literal['Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Limited Size or Shade Availability', 
                    'Unresponsive or Generic Customer Support', 'Website or App Bugs', 'Confusing Loyalty or Discount Systems', 
                    'Complicated Returns or Exchanges', 'Customs and Import Charges', 'Difficult Product Discovery', 
                    'Damaged or Incorrect Items']] = dspy.OutputField()

In [24]:
print(nps_data[0])
print(nps_data[0]['topics'])
print(wrap_text(nps_data[0]['comment'], width=72))

{'topics': ['Limited Size or Shade Availability'], 'comment': "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?"}
['Limited Size or Shade Availability']
Absolutely frustrated! Every time I find something I love, it's sold out
in my size. What's the point of having a wishlist if nothing is ever
available?


In [25]:
nps_topic_module = dspy.ChainOfThought(NPSTopic)  # dspy CoT add a reasoning field to the prediction
response = nps_topic_module(comment = "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?")
print(response)
print(response.reasoning)
print(response.answer)


Prediction(
    reasoning='The user is expressing frustration because the items they are interested in are frequently unavailable in their size, which they find disappointing when using the wishlist feature.',
    answer=['Limited Size or Shade Availability']
)
The user is expressing frustration because the items they are interested in are frequently unavailable in their size, which they find disappointing when using the wishlist feature.
['Limited Size or Shade Availability']


In [26]:
dspy.inspect_history(n = 1)






[2026-05-11T23:07:25.659589]

System message:

Your input fields are:
1. `comment` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (list[Literal['Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Limited Size or Shade Availability', 'Unresponsive or Generic Customer Support', 'Website or App Bugs', 'Confusing Loyalty or Discount Systems', 'Complicated Returns or Exchanges', 'Customs and Import Charges', 'Difficult Product Discovery', 'Damaged or Incorrect Items']]):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## comment ## ]]
{comment}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}        # note: the value you produce must adhere to the JSON schema: {\"type\": \"array\", \"items\": {\"type\": \"string\", \"enum\": [\"Slow or Unreliable Shipping\", \"Inaccurate Product Descrip

In [27]:
nps_df = pd.DataFrame(nps_data)
# Net effect: each row gets a 1-based id (1, 2, 3, …, N) in a new column.
nps_df['id'] = list(map(lambda x: x + 1, range(nps_df.shape[0])))

In [28]:
tmp = []

for rec in tqdm.tqdm(nps_df.to_dict('records')):
    response = nps_topic_module(comment = rec['comment'])  # dspy.InputField comment
    res = {
        'id': rec['id'],
        'model_topics': response.answer
    }

    tmp.append(res)

100%|██████████| 105/105 [02:44<00:00,  1.57s/it]


In [29]:
ini_model_topics_df = pd.DataFrame(tmp)

In [30]:
ini_model_topics_df

,id,model_topics
0,1,[Limited Size or Shade Availability]
1,2,[Difficult Product Discovery]
2,3,[Inaccurate Product Descriptions or Photos]
3,4,"[Confusing Loyalty or Discount Systems, Unresp..."
4,5,[Website or App Bugs]
...,...,...
100,101,[Customs and Import Charges]
101,102,"[Confusing Loyalty or Discount Systems, Compli..."
102,103,"[Limited Size or Shade Availability, Unrespons..."
103,104,"[Difficult Product Discovery, Inaccurate Produ..."


In [31]:
nps_df = nps_df.merge(ini_model_topics_df)

In [32]:
# show head
nps_df.sample(5).to_dict('records')

[{'topics': ['Complicated Returns or Exchanges'],
  'comment': 'Have to pay return shipping for defective products. This should be covered by the company, not the customer.',
  'id': 40,
  'model_topics': ['Complicated Returns or Exchanges']},
 {'topics': ['Website or App Bugs'],
  'comment': "Cart keeps emptying itself when I switch between pages. I've had to start over 5 times trying to place one order.",
  'id': 16,
  'model_topics': ['Website or App Bugs']},
 {'topics': ['Inaccurate Product Descriptions or Photos'],
  'comment': 'Bronzer shade appeared warm-toned online but arrived cool-toned. Photography and color accuracy are terrible.',
  'id': 47,
  'model_topics': ['Inaccurate Product Descriptions or Photos']},
 {'topics': ['Limited Size or Shade Availability'],
  'comment': "My shade is ALWAYS out of stock. I've been trying to reorder my foundation for 3 weeks now. Time to find a new brand that actually keeps products in stock.",
  'id': 6,
  'model_topics': ['Limited Size or

In [33]:
def compare_topics(l1, l2):
    l1_fmt = ', '.join(sorted(l1))
    l2_fmt = ', '.join(sorted(l2))
    if l1_fmt == l2_fmt: 
        return 1 
    return 0


nps_df['model_accuracy'] = list(map(
    compare_topics,
    nps_df.topics,
    nps_df.model_topics
))

In [34]:
round(100*nps_df.model_accuracy.mean(), 2)

np.float64(86.67)

In [35]:
import random
random.random()

0.7924162708348153

In [36]:
trainset = []
valset = []
for rec in nps_data: 
    if random.random() <= 0.5:
        trainset.append(
            dspy.Example(
                comment = rec['comment'],
                answer = rec['topics']
            ).with_inputs('comment')
        )
    else: 
        valset.append(
            dspy.Example(
                comment = rec['comment'],
                answer = rec['topics']
            ).with_inputs('comment')
        )

In [37]:
# tp = dspy.MIPROv2(metric=dspy.evaluate.answer_exact_match, auto="light", num_threads=24)

In [40]:
def list_exact_match(example, pred, trace=None, pred_name=None, pred_trace=None):
    """Score topic-list predictions, with feedback for GEPA's reflector.

    Returns a ``dspy.Prediction(score=..., feedback=...)``:
      * ``score`` is 1.0 if the predicted topic set equals the gold set, else 0.0.
        BootstrapFewShotWithRandomSearch and MIPROv2 read this field.
      * ``feedback`` is a one-sentence diagnosis of the failure mode (missing
        topics, hallucinated topics, type mismatch). GEPA's ``reflection_lm``
        consumes this to rewrite the prompt; the other optimisers ignore it.
    """
    try:
        pred_answer = pred.answer
        expected_answer = example.answer

        if isinstance(pred_answer, list) and isinstance(expected_answer, list):
            pred_set = set(pred_answer)
            gold_set = set(expected_answer)
            score = 1.0 if pred_set == gold_set else 0.0
            if score == 1.0:
                feedback = "Correct: predicted topic set matches the gold set."
            else:
                missing = gold_set - pred_set
                extra = pred_set - gold_set
                parts = []
                if missing:
                    parts.append(f"missing topics: {sorted(missing)}")
                if extra:
                    parts.append(f"hallucinated topics not in gold: {sorted(extra)}")
                feedback = (
                    f"Incorrect. Gold: {sorted(gold_set)}. Predicted: {sorted(pred_set)}. "
                    + "; ".join(parts) + "."
                )
        else:
            score = 1.0 if pred_answer == expected_answer else 0.0
            verdict = "Correct" if score else "Incorrect"
            feedback = (
                f"{verdict}: predicted {pred_answer!r}, expected {expected_answer!r}."
            )
        return dspy.Prediction(score=score, feedback=feedback)
    except Exception as e:
        return dspy.Prediction(score=0.0, feedback=f"Metric error: {e}")

In [41]:
reflection_lm = dspy.LM(REFLECTION_MODEL_CANDIDATES[0], temperature=1.0, max_tokens=32768)

# GEPA metric must accept five arguments: (gold, pred, trace, pred_name, pred_trace)
gepa_tele_prompter = dspy.GEPA(
    metric=list_exact_match,
    auto="light",
    num_threads=24,
    reflection_lm=reflection_lm,
    track_stats=True,
)

In [43]:
opt_nps_topic_model = gepa_tele_prompter.compile(
    nps_topic_module,
    trainset=trainset,
    valset=valset,
)

2026/05/11 23:22:24 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 536 metric calls of the program. This amounts to 5.10 full evals on the train+val set.
2026/05/11 23:22:24 INFO dspy.teleprompt.gepa.gepa: Using 39 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/536 [00:00<?, ?rollouts/s]2026/05/11 23:22:27 INFO dspy.evaluate.evaluate: Average Metric: 32.0 / 39 (82.1%)
2026/05/11 23:22:27 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.8205128205128205 over 39 / 39 examples
GEPA Optimization:   7%|▋         | 39/536 [00:03<00:43, 11.44rollouts/s]2026/05/11 23:22:27 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score

Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00,  3.47it/s]

2026/05/11 23:22:28 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/05/11 23:22:28 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2026/05/11 23:22:28 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
GEPA Optimization:   8%|▊         | 42/536 [00:04<00:53,  9.30rollouts/s]2026/05/11 23:22:28 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 0.8205128205128205



Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:00<00:00,  4.10it/s]

2026/05/11 23:22:29 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)


...


Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00,  3.76it/s]

2026/05/11 23:29:36 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/05/11 23:29:36 INFO dspy.teleprompt.gepa.gepa: Iteration 34: All subsample scores perfect. Skipping.
2026/05/11 23:29:36 INFO dspy.teleprompt.gepa.gepa: Iteration 34: Reflective mutation did not propose a new candidate
GEPA Optimization:  98%|█████████▊| 525/536 [07:11<00:04,  2.47rollouts/s]2026/05/11 23:29:36 INFO dspy.teleprompt.gepa.gepa: Iteration 35: Selected program 1 score: 0.8717948717948718



Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:00<00:00,  3.84it/s] 

2026/05/11 23:29:36 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)


2026/05/11 23:29:54 INFO dspy.teleprompt.gepa.gepa: Iteration 35: Proposed new text for predict: Your task is to classify Net Promoter Score (NPS) comments into appropriate topics.

### Inputs
- `comment`: The customer's feedback text.

### Outputs
Produce your response with the exact following headers:
### reasoning
Explain the primary underlying issue in the comment. Determine the root cause of the customer's frustration.

### answer
Provide a Python-formatted list of strings representing the predicted topic(s), for example: ['Website or App Bugs']. Only include the most specific, primary root cause(s) and avoid over-tagging.

### Known Topics
Your possible topics include (but may not be limited to):
- 'Slow or Unreliable Shipping'
- 'Website or App Bugs'
- 'Unresponsive or Generic Customer Support'
- 'Damaged or Incorrect Items'
- 'Customs and Import Charges'
- 'Complicated Returns or Exchanges'

### Classification Rules & Domain Constraints
- **Focus on the Root Cause**: Often, a c

In [44]:
opt_nps_topic_model(comment = "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?"
)

Prediction(
    reasoning='The customer is expressing frustration because the items they are interested in are frequently unavailable in their size. This indicates a problem with inventory management or product availability, specifically concerning sizing.',
    answer=['Limited Size or Shade Availability']
)

In [45]:
dspy.inspect_history(n = 1)





[2026-05-11T23:30:00.173913]

System message:

Your input fields are:
1. `comment` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (list[Literal['Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Limited Size or Shade Availability', 'Unresponsive or Generic Customer Support', 'Website or App Bugs', 'Confusing Loyalty or Discount Systems', 'Complicated Returns or Exchanges', 'Customs and Import Charges', 'Difficult Product Discovery', 'Damaged or Incorrect Items']]):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## comment ## ]]
{comment}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}        # note: the value you produce must adhere to the JSON schema: {\"type\": \"array\", \"items\": {\"type\": \"string\", \"enum\": [\"Slow or Unreliable Shipping\", \"Inaccurate Product Descrip

In [46]:
tmp = []

for e in tqdm.tqdm(valset):
    comment = e.comment 
    prev_resp = nps_topic_module(comment = comment) 
    new_resp = opt_nps_topic_model(comment = comment)

    tmp.append(
        {
        'comment': comment,
        'gold_answer': e.answer,
        'prev_answer': prev_resp.answer,
        'new_answer': new_resp.answer
        }
    )

100%|██████████| 39/39 [01:17<00:00,  1.99s/it]


In [47]:
cmp_df = pd.DataFrame(tmp)

In [48]:
def list_exact_match_raw(expected_answer, pred_answer, trace=None):
    """Custom metric for comparing lists of topics"""
    try:
        # Convert to sets for order-independent comparison
        if isinstance(pred_answer, list) and isinstance(expected_answer, list):
            return set(pred_answer) == set(expected_answer)
        else:
            return pred_answer == expected_answer
    except Exception as e:
        print(f"Error in metric: {e}")
        return False

In [49]:
cmp_df['prev_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.gold_answer,
    cmp_df.prev_answer))

cmp_df['new_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.gold_answer,
    cmp_df.new_answer))

In [51]:
cmp_df[['prev_accuracy', 'new_accuracy']].mean()*100

prev_accuracy    84.615385
new_accuracy     79.487179
dtype: float64

### dspy.BootstrapFewShotWithRandomSearch

In [ ]:
tp2 = dspy.BootstrapFewShotWithRandomSearch(list_exact_match, num_threads=24, max_bootstrapped_demos = 10)

In [ ]:
opt2_nps_topic_model =  tp2.compile(
    nps_topic_module, 
    trainset=trainset, 
    valset=valset)

In [ ]:
tmp = []

for e in tqdm.tqdm(valset):
    comment = e.comment 
    prev_resp = nps_topic_module(comment = comment) 
    new_resp = opt_nps_topic_model(comment = comment)
    new_reason_resp = opt2_nps_topic_model(comment = comment)

    tmp.append(
        {
        'comment': comment,
        'gold_answer': e.answer,
        'prev_answer': prev_resp.answer,
        'new_answer': new_resp.answer,
        'new_reason_answer': new_reason_resp.answer
        }
    )

In [ ]:
cmp_df = pd.DataFrame(tmp)

In [ ]:
cmp_df['prev_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.gold_answer,
    cmp_df.prev_answer))

cmp_df['new_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.gold_answer,
    cmp_df.new_answer))

cmp_df['new_reason_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.gold_answer,
    cmp_df.new_reason_answer))

In [ ]:
cmp_df[['prev_accuracy', 'new_accuracy', 'new_reason_accuracy']].mean()*100

In [ ]:
opt2_nps_topic_model(comment = "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?"
)

In [ ]:
dspy.inspect_history(n = 1)